## DNS Tunneling Detection ML Pipeline — Stages 9, 10 (Part 2), 11 & 12

---

| Stage | Title | Deliverable |
|-------|-------|-------------|
| **9** | Advanced ML / DL Modeling | SVM with RBF Kernel + ANN (MLP) with Backpropagation |
| **10 (Part 2)** | Advanced Hyperparameter Tuning | GridSearchCV for SVM (`C`, `gamma`) and ANN (`hidden_layer_sizes`, `learning_rate_init`, `alpha`) |
| **11** | Validation & Comparative Evaluation | Unified ROC-AUC, Precision-Recall, Confusion Matrices, F1 across CART vs SVM vs ANN |
| **12** | Real-World Use Case & Limitations | Enterprise NIDS deployment architecture + Adversarial ML evasion analysis |

### Pipeline Position
$$\underbrace{X_{\text{train\_eng}}, X_{\text{test\_eng}}}_{\text{stage 7,8,10's 18 Features}} \longrightarrow \underbrace{\text{SVM (RBF)} + \text{ANN (MLP)}}_{\text{Stage 9}} \longrightarrow \underbrace{\text{GridSearchCV}}_{\text{Stage 10}} \longrightarrow \underbrace{\text{CART vs SVM vs ANN}}_{\text{Stage 11}} \longrightarrow \underbrace{\text{NIDS Deployment}}_{\text{Stage 12}}$$


In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.size': 11, 'figure.titlesize': 14, 'axes.labelsize': 12})

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DIR = Path('data/processed')
FIGURES_DIR = Path('figures')

print("Environment ready.")


---
## Loading Upstream Deliverables

We ingest **baseline model's feature matrices** (18 features after multicollinearity elimination and feature construction) and the **CART baseline model** that serves as our benchmark.


In [ ]:
# Load baseline feature matrices
X_train = pd.read_csv(PROCESSED_DIR / 'X_train_engineered.csv')
y_train = pd.read_csv(PROCESSED_DIR / 'y_train_bal.csv').squeeze()
X_test  = pd.read_csv(PROCESSED_DIR / 'X_test_engineered.csv')
y_test  = pd.read_csv(PROCESSED_DIR / 'y_test.csv').squeeze()

# Load CART baseline
dt_tuned = joblib.load(PROCESSED_DIR / 'cart_tuned_model.joblib')
y_pred_dt = dt_tuned.predict(X_test)

print("=== Upstream Data Loaded ===")
print(f"X_train: {X_train.shape}  |  Balance: {dict(y_train.value_counts())}")
print(f"X_test:  {X_test.shape}  |  Balance: {dict(y_test.value_counts())}")
print(f"Features: {X_train.columns.tolist()}")
print(f"\nCART Baseline loaded (depth={dt_tuned.get_depth()}, F1={f1_score(y_test, y_pred_dt):.4f})")


---
## Stage 9: Advanced ML Modeling

### Model 1: Support Vector Machine (SVM) with RBF Kernel

#### Why SVM for DNS Tunneling?
DNS tunneling features (entropy, query length, record type frequencies) occupy a complex, **non-linearly separable** manifold. Benign and malicious queries overlap significantly along individual linear projections — stealthy, low-volume exfiltration traffic deliberately mimics normal patterns along most features.

#### The Kernel Trick: Implicit Infinite-Dimensional Projection
The **Radial Basis Function (RBF) kernel** projects feature vectors into an infinite-dimensional Hilbert space where a linear hyperplane achieves clean class separation, *without explicitly computing the high-dimensional coordinates*:

$$K(x, x') = \exp\left(-\gamma \|x - x'\|^2\right)$$

where:
- $\gamma > 0$ controls the **radius of influence** of each support vector. Large $\gamma$ creates tight, complex decision boundaries (risk of overfitting); small $\gamma$ produces smooth, generalised boundaries.
- $C$ (regularisation parameter) balances **margin maximisation** against **slack variable misclassification penalties**. Large $C$ enforces hard margins (memorises noise); small $C$ tolerates errors for wider margins.

#### Computational Scaling Note
SVM training complexity is $O(n^2)$ to $O(n^3)$ in the number of samples. With $158{,}598$ training rows, full-data GridSearchCV is prohibitively expensive. We employ a **balanced stratified subsample** for hyperparameter tuning and a larger subsample for final model training.


In [ ]:
def stratified_subsample(X, y, n_samples, seed=42):
    rng = np.random.RandomState(seed)
    n_per_class = n_samples // 2
    idx_0 = rng.choice(np.where(y == 0)[0], size=min(n_per_class, (y==0).sum()), replace=False)
    idx_1 = rng.choice(np.where(y == 1)[0], size=min(n_per_class, (y==1).sum()), replace=False)
    idx = np.concatenate([idx_0, idx_1])
    rng.shuffle(idx)
    return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)

SVM_TUNE_SAMPLE = 15000
SVM_FINAL_SAMPLE = 40000

X_tune_svm, y_tune_svm = stratified_subsample(X_train, y_train, SVM_TUNE_SAMPLE)
print(f"SVM tuning subsample: {X_tune_svm.shape} (balanced)")


### Stage 10 (Part 2): SVM Hyperparameter Tuning via GridSearchCV

| Hyperparameter | Effect | Grid Values |
|---|---|---|
| `C` | Regularisation penalty — controls margin width vs error tolerance | `[0.1, 1.0, 10.0]` |
| `gamma` | RBF kernel bandwidth — controls support vector influence radius | `['scale', 0.01, 0.1]` |
| `kernel` | Kernel function type | `['rbf']` (fixed) |

**Method**: 3-Fold Stratified Cross-Validation on balanced subsample.


In [ ]:
param_grid_svm = {
    'C': [0.1, 1.0, 10.0],
    'gamma': ['scale', 0.01, 0.1],
    'kernel': ['rbf']
}

cv_svm = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_svm = GridSearchCV(
    SVC(random_state=RANDOM_STATE, probability=True),
    param_grid_svm, cv=cv_svm, scoring='f1',
    n_jobs=-1, verbose=1, return_train_score=True
)

print("Starting SVM GridSearchCV...")
t0 = time.time()
grid_svm.fit(X_tune_svm, y_tune_svm)
svm_tune_time = time.time() - t0

print(f"\nSVM GridSearchCV completed in {svm_tune_time:.1f}s")
print(f"Best CV F1: {grid_svm.best_score_:.4f}")
print(f"Best params: {grid_svm.best_params_}")


In [ ]:
# Train final SVM on larger subsample with best hyperparameters
X_final_svm, y_final_svm = stratified_subsample(X_train, y_train, SVM_FINAL_SAMPLE)
print(f"Training final SVM on {X_final_svm.shape[0]:,} samples with best params...")

t0 = time.time()
svm_final = SVC(**grid_svm.best_params_, random_state=RANDOM_STATE, probability=True)
svm_final.fit(X_final_svm, y_final_svm)
svm_train_time = time.time() - t0
print(f"Final SVM trained in {svm_train_time:.1f}s")

y_pred_svm = svm_final.predict(X_test)
y_prob_svm = svm_final.predict_proba(X_test)[:, 1]

print(f"\n--- SVM (RBF) Test Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_svm):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_svm):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_svm):.4f}")
print(f"\n{classification_report(y_test, y_pred_svm, target_names=['Benign', 'Malicious'])}")


---
### Model 2: Artificial Neural Network (Multi-Layer Perceptron — MLP)

#### Network Architecture
The MLP implements a fully connected feed-forward neural network trained via **Backpropagation** with the **Adam** optimiser:

- **Input Layer**: 18 neurons (one per engineered feature from Harsh)
- **Hidden Layer 1**: Neurons with **ReLU** activation:
  $$f(z) = \max(0, z)$$
  *Justification*: ReLU solves the **vanishing gradient problem** that plagues sigmoid/tanh in deep networks, enabling efficient gradient flow.
- **Hidden Layer 2**: Neurons with **ReLU** activation (feature abstraction layer)
- **Output Layer**: 1 neuron with **Sigmoid** (logistic) activation:
  $$\sigma(z) = \frac{1}{1 + e^{-z}}$$
  Outputs a calibrated posterior probability $P(\text{Malicious} \mid x) \in [0.0, 1.0]$.

#### Loss Function: Binary Cross-Entropy (Log-Loss)
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

Minimised via **Backpropagation** — the chain-rule-based gradient computation algorithm that propagates output errors backward through each layer to update weights:
$$w \leftarrow w - \eta \frac{\partial \mathcal{L}}{\partial w}$$

where $\eta$ is the learning rate.


### Stage 10 (Part 2): ANN Hyperparameter Tuning via GridSearchCV

| Hyperparameter | Effect | Grid Values |
|---|---|---|
| `hidden_layer_sizes` | Network topology (neurons per hidden layer) | `[(16, 8), (32, 16), (64, 32)]` |
| `learning_rate_init` | Initial step size for Adam gradient descent | `[0.001, 0.01]` |
| `alpha` | $L_2$ regularisation penalty on weight norms | `[0.0001, 0.001]` |
| `activation` | Hidden layer activation function | `['relu']` (fixed) |

**Method**: 3-Fold Stratified Cross-Validation on full balanced training set.


In [ ]:
param_grid_ann = {
    'hidden_layer_sizes': [(16, 8), (32, 16), (64, 32)],
    'learning_rate_init': [0.001, 0.01],
    'alpha': [0.0001, 0.001],
    'activation': ['relu']
}

cv_ann = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

grid_ann = GridSearchCV(
    MLPClassifier(max_iter=300, early_stopping=True, validation_fraction=0.1,
                  random_state=RANDOM_STATE, solver='adam'),
    param_grid_ann, cv=cv_ann, scoring='f1',
    n_jobs=-1, verbose=1, return_train_score=True
)

print("Starting ANN GridSearchCV (full training set)...")
t0 = time.time()
grid_ann.fit(X_train, y_train)
ann_tune_time = time.time() - t0

print(f"\nANN GridSearchCV completed in {ann_tune_time:.1f}s")
print(f"Best CV F1: {grid_ann.best_score_:.4f}")
print(f"Best params: {grid_ann.best_params_}")


In [ ]:
ann_tuned = grid_ann.best_estimator_

y_pred_ann = ann_tuned.predict(X_test)
y_prob_ann = ann_tuned.predict_proba(X_test)[:, 1]

print(f"--- ANN (MLP) Test Performance ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ann):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_ann):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_ann):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_ann):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_ann):.4f}")
print(f"\n{classification_report(y_test, y_pred_ann, target_names=['Benign', 'Malicious'])}")


---
## Stage 11: Validation & Comparative Multi-Model Evaluation

### The IA Guideline
> *"Accuracy alone is generally insufficient when class distributions are uneven."*

Our test set contains **33,986 Benign** queries and only **85 Malicious** queries ($0.25\%$ attack prevalence). The **accuracy paradox** means a degenerate model predicting 100% Benign achieves **99.75% accuracy** yet has **0% Recall** — completely useless for security.

### Cost Asymmetry in Cybersecurity
| Error Type | Consequence | Severity |
|---|---|---|
| **False Negative (FN)** | Actual tunneling attack bypasses detection → **Catastrophic data breach** | 🔴 Critical |
| **False Positive (FP)** | Benign query flagged as malicious → **SOC analyst investigates** | 🟡 Manageable |

### Evaluation Metrics
1. **Recall** (Sensitivity): $\frac{TP}{TP + FN}$ — Must be maximised to catch every exfiltration attempt.
2. **Precision**: $\frac{TP}{TP + FP}$ — Alert fidelity to prevent alarm fatigue.
3. **F1-Score**: $2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$ — Harmonic mean; proves stability.
4. **ROC-AUC**: Area under the TPR vs FPR curve across all thresholds $\tau \in [0, 1]$.
5. **PR-AUC (Average Precision)**: Critical supplementary metric for extreme imbalance.


In [ ]:
# Unified Comparative ROC & Precision-Recall Curves
y_prob_dt = dt_tuned.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROC Curve
for name, y_prob, color, ls in [
    ('CART Baseline', y_prob_dt, '#ff7f0e', '--'),
    ('SVM (RBF)', y_prob_svm, '#1f77b4', '-'),
    ('ANN (MLP)', y_prob_ann, '#d62728', '-')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    axes[0].plot(fpr, tpr, color=color, linestyle=ls, linewidth=2,
                 label=f'{name} (AUC = {auc_val:.4f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier')
axes[0].set_xlabel('False Positive Rate (FPR)')
axes[0].set_ylabel('True Positive Rate (TPR / Recall)')
axes[0].set_title('Comparative ROC Curves', fontsize=13, fontweight='bold')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# Precision-Recall Curve
for name, y_prob, color, ls in [
    ('CART Baseline', y_prob_dt, '#ff7f0e', '--'),
    ('SVM (RBF)', y_prob_svm, '#1f77b4', '-'),
    ('ANN (MLP)', y_prob_ann, '#d62728', '-')
]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    axes[1].plot(rec, prec, color=color, linestyle=ls, linewidth=2,
                 label=f'{name} (AP = {ap:.4f})')

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Comparative Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Stage 11: Unified Tri-Model Comparative Evaluation', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage11_comparative_roc_pr_curves.png', dpi=300)
plt.show()


In [ ]:
# Tri-Panel Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

models = [
    ('CART Baseline', y_pred_dt, 'Oranges'),
    ('SVM (RBF)', y_pred_svm, 'Blues'),
    ('ANN (MLP)', y_pred_ann, 'Greens')
]
for ax, (name, y_pred, cmap) in zip(axes, models):
    cm = confusion_matrix(y_test, y_pred)
    f1_val = f1_score(y_test, y_pred)
    rec_val = recall_score(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Attack']).plot(
        ax=ax, cmap=cmap, values_format=',d')
    ax.set_title(f'{name}\nF1={f1_val:.4f} | Recall={rec_val:.4f}', fontsize=11, fontweight='bold')

plt.suptitle('Stage 11: Tri-Model Confusion Matrices (Test Set, N=34,071)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage11_tri_model_confusion_matrices.png', dpi=300)
plt.show()


In [ ]:
# Comprehensive Metric Comparison Table
metrics = {}
for name, y_pred, y_prob in [
    ('CART Baseline', y_pred_dt, y_prob_dt),
    ('SVM (RBF)', y_pred_svm, y_prob_svm),
    ('ANN (MLP)', y_pred_ann, y_prob_ann)
]:
    metrics[name] = {
        'Accuracy': f"{accuracy_score(y_test, y_pred):.4f}",
        'Precision': f"{precision_score(y_test, y_pred):.4f}",
        'Recall': f"{recall_score(y_test, y_pred):.4f}",
        'F1-Score': f"{f1_score(y_test, y_pred):.4f}",
        'ROC-AUC': f"{roc_auc_score(y_test, y_prob):.4f}",
        'AP (PR-AUC)': f"{average_precision_score(y_test, y_prob):.4f}"
    }

comparison_df = pd.DataFrame(metrics).T
comparison_df.index.name = 'Model'

print("=" * 80)
print("  COMPREHENSIVE TRI-MODEL COMPARISON (TEST SET, N=34,071)")
print("=" * 80)
print(comparison_df.to_string())

comparison_df.to_csv(PROCESSED_DIR / 'stage11_model_comparison.csv')
print(f"\nSaved: {PROCESSED_DIR / 'stage11_model_comparison.csv'}")


In [ ]:
# Metric Comparison Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metric_names))
width = 0.25
colors = ['#ff7f0e', '#1f77b4', '#d62728']

for i, (model_name, vals) in enumerate(metrics.items()):
    bar_vals = [float(vals[m]) for m in metric_names]
    ax.bar(x + i * width, bar_vals, width, label=model_name, color=colors[i])

ax.set_ylabel('Score')
ax.set_title('Stage 11: Comprehensive Metric Comparison Across All Models', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(metric_names)
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'stage11_metric_comparison_bar.png', dpi=300)
plt.show()


---
## Stage 12: Real-World Use Case, Enterprise Deployment & Limitations

### 1. Production Deployment: Inline Network Intrusion Detection System (NIDS)

#### Architecture
The trained pipeline is packaged into a containerised microservice deployed as a **passive DNS traffic analyser** within enterprise network infrastructure:

```
                    ┌─────────────────────────────────────┐
                    │        Enterprise Network           │
                    │                                     │
  [User Workstation]──→ [Internal DNS Forwarder] ──→ [Internet]
                    │          │                          │
                    │          │  Mirror/TAP              │
                    │          ▼                          │
                    │   ┌──────────────┐                  │
                    │   │  ML Scoring  │                  │
                    │   │  Microservice│                  │
                    │   │              │                  │
                    │   │ 1. Extract   │                  │
                    │   │    Features  │                  │
                    │   │ 2. Scale     │                  │
                    │   │ 3. Predict   │                  │
                    │   │ 4. Score     │                  │
                    │   └──────┬───────┘                  │
                    │          │                          │
                    │          ▼                          │
                    │   [Firewall API / SIEM Alert]       │
                    └─────────────────────────────────────┘
```

#### Operational Flow
1. **Packet Capture**: Zeek/Suricata/dnstap sensors mirror DNS queries from internal resolvers (BIND9, Unbound, CoreDNS).
2. **Feature Extraction**: Each query is vectorised into Harsh's 18 engineered features (entropy, length, record type frequencies, network diversity).
3. **Scaling**: Apply the serialised `preprocessor_scaler.joblib` fitted on training data.
4. **Inference**: Score via the serialised model (`.joblib`). Queries exceeding threshold $P(\text{Malicious}) \ge \tau$ are flagged.
5. **Response**: Flagged queries trigger SIEM alerts, and optionally, firewall API calls to block/quarantine the source IP.

### 2. Critical Limitations: Adversarial Machine Learning

#### Limitation 1: Entropy Evasion
If adversaries discover that our model detects high Shannon entropy (from Base64/Base32 encoding), they can adapt by:
- **Dictionary-word encoding**: Embedding exfiltrated data into valid English words (e.g., `apple.banana.cherry.evil.com`), artificially lowering entropy to $H \approx 2.5$ (indistinguishable from benign queries).
- **Markov chain phonetic generators**: Producing subdomains that mimic natural language character distributions.

#### Limitation 2: Low-and-Slow Exfiltration
Adversaries can reduce query frequency below detection thresholds:
- A single DNS query every $5{-}10$ minutes carries $< 63$ bytes of payload per query.
- At this rate, exfiltrating a $1$ MB credential database would take $\sim 11$ days, but would be statistically invisible against normal background DNS noise.

#### Limitation 3: Concept Drift
DNS traffic patterns evolve as enterprises adopt new services, CDN configurations, and cloud infrastructure. Models trained on 2020-era traffic may degrade on 2025+ patterns without periodic retraining.

#### Mitigation Strategy: Defense-in-Depth
No single ML model provides comprehensive coverage. The production system should combine:
1. **Lexical statistical models** (this pipeline — entropy, length, record types)
2. **Temporal sequence analysis** (LSTM/Transformer over time-series query frequencies)
3. **IP reputation telemetry** (threat intelligence feeds, geolocation anomaly detection)
4. **Periodic model retraining** on fresh labelled captures to combat concept drift

### 3. Final Recommendation

Based on the comparative evaluation across CART Baseline, SVM (RBF), and ANN (MLP), we select the model with the optimal **Recall-Precision-F1 trade-off** for production deployment. In cybersecurity applications, **maximising Recall** (minimising missed attacks) takes precedence over Precision, provided alarm fatigue remains manageable.


In [ ]:
# Save all models and results
joblib.dump(svm_final, PROCESSED_DIR / 'svm_tuned_model.joblib')
joblib.dump(ann_tuned, PROCESSED_DIR / 'ann_tuned_model.joblib')

with open(PROCESSED_DIR / 'svm_best_hyperparameters.json', 'w') as f:
    json.dump(grid_svm.best_params_, f, indent=2, default=str)
with open(PROCESSED_DIR / 'ann_best_hyperparameters.json', 'w') as f:
    json.dump(grid_ann.best_params_, f, indent=2, default=str)

pd.DataFrame(grid_svm.cv_results_).to_csv(PROCESSED_DIR / 'svm_gridsearch_results.csv', index=False)
pd.DataFrame(grid_ann.cv_results_).to_csv(PROCESSED_DIR / 'ann_gridsearch_results.csv', index=False)

# Determine winner
f1s = {
    'CART Baseline': f1_score(y_test, y_pred_dt),
    'SVM (RBF)': f1_score(y_test, y_pred_svm),
    'ANN (MLP)': f1_score(y_test, y_pred_ann)
}
winner = max(f1s, key=f1s.get)

print("=" * 70)
print("  ALL STAGES COMPLETE")
print("=" * 70)
print(f"\n  Models saved: svm_tuned_model.joblib, ann_tuned_model.joblib")
print(f"  SVM best params: {grid_svm.best_params_}")
print(f"  ANN best params: {grid_ann.best_params_}")
print(f"\n  Test F1 Scores: {f1s}")
print(f"  RECOMMENDED PRODUCTION MODEL: {winner} (F1 = {f1s[winner]:.4f})")
print("=" * 70)
